# Google client

#### 1. Dependencies

In [1]:
%pip install --quiet google-genai pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import pandas as pd
from google import genai

#### 2. Configuration

In [3]:
MODEL_NAME = "gemini-2.5-flash"
MODEL_SHORT = "Gemini-2.5-Flash"

def load_api_key(path, name):
    for line in open(path).read().strip().splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise KeyError(f"Klucz '{name}' nie znaleziony w {path}")

API_KEY = load_api_key("./api_key.txt", "google")

PROMPTS_PATH = "./prompts.tsv"
OUTPUT_DIR = f"./responses_{MODEL_SHORT}"
RESPONSES_PATH = f"{OUTPUT_DIR}/responses.tsv"

POLITE_DELAY = 7.0

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 3. Load prompts (with resume)

In [4]:
prompts_df = pd.read_csv(PROMPTS_PATH, sep="\t")

try:
    df = pd.read_csv(RESPONSES_PATH, sep="\t")
    df.loc[df["response"].astype(str).str.startswith("EXCEPTION:"), "response"] = pd.NA
    done = df["response"].notna().sum()
    print(f"Wznowiono: {done}/{len(df)} wykonanych")
except FileNotFoundError:
    df = prompts_df.copy()
    df["response"] = pd.NA
    print(f"Start od poczatku: {len(df)} promptow do wyslania")

Wznowiono: 0/12 wykonanych


#### 4. Send loop

In [ ]:
client = genai.Client(api_key=API_KEY)

MAX_RETRIES = 4
RETRY_BASE_DELAY = 15.0

def is_transient(exc):
    msg = str(exc).upper()
    return any(kw in msg for kw in ("503", "UNAVAILABLE", "OVERLOAD", "TRY AGAIN", "RESOURCE_EXHAUSTED"))

def send_one(prompt):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            return response.text or ""
        except Exception as exc:
            if is_transient(exc) and attempt < MAX_RETRIES - 1:
                wait = RETRY_BASE_DELAY * (2 ** attempt)
                print(f"    503/przeciazenie, czekam {wait:.0f}s (proba {attempt + 1}/{MAX_RETRIES})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Nieosiagalne")

pending = df[df["response"].isna()].index.tolist()
print(f"Do wyslania: {len(pending)}")
print(f"Szacowany czas: {len(pending) * POLITE_DELAY / 60:.1f} min")

for i in pending:
    item_id = df.at[i, "id"]
    try:
        text = send_one(df.at[i, "prompt"])
        df.at[i, "response"] = text
        print(f"  [{pending.index(i) + 1}/{len(pending)}] {item_id}: {len(text):,} chars")
    except Exception as exc:
        df.at[i, "response"] = pd.NA
        print(f"  [{pending.index(i) + 1}/{len(pending)}] {item_id}: BLAD - {exc}")

    df.to_csv(RESPONSES_PATH, sep="\t", index=False)
    time.sleep(POLITE_DELAY)

print("Gotowe!")

Do wyslania: 12
Szacowany czas: 1.4 min
    503/przeciazenie, czekam 15s (proba 1/4)
    503/przeciazenie, czekam 30s (proba 2/4)
    503/przeciazenie, czekam 60s (proba 3/4)
  [1/12] zh--adam_grant_how_to_stop_languishing_and_start_finding_flow: BLAD - 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 37.95515388s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure',